# Heat-Sink Geometry Variation Study

This notebook documents how the heat-sink geometry variants were created, why the geometry study was introduced, the ANSYS conditions used, the raw maximum-temperature results, and the conversion to estimated MOSFET junction temperature.

The purpose is to keep the **simulation method and raw geometry data** separate from the higher-level engineering interpretation contained in `../results/engineering_assessment.ipynb`.


## 1. Baseline Geometry

The original heat sink used a base size of:

- **80 mm × 50 mm × 5 mm**
- **ANSYS-reported baseline convection surface area:** **0.0533 m²**

Initial simulations showed only a very small difference between aluminium and copper at the baseline MOSFET heat load. This suggested that the original heat sink was relatively large for the low heat input, so smaller geometry variants were introduced to test whether geometry would become a more influential design variable.


The later natural-convection sensitivity uses geometry-derived idealised exposed areas for all three designs. Those sensitivity areas are kept separate from this ANSYS-reported baseline area used in the analytical validation.

## 2. Geometry Variants

Two reduced heat-sink variants were investigated.

| Geometry | Base dimensions | Modification |
|---|---|---|
| Original | 80 mm × 50 mm × 5 mm | Baseline geometry |
| Variant A | 60 mm × 50 mm × 5 mm | 20 mm removed from the side opposite the TIM; two fins removed |
| Variant B | 40 mm × 50 mm × 5 mm | Further reduction in heat-sink length and fin count |

The reported convection surface area of the **60 mm** variant was approximately **0.041 m²**, compared with **0.0533 m²** for the original heat sink.

> Physical dimensions are used as the primary geometry identifier in this notebook to avoid ambiguity between `Geometry_1` and `Geometry_2` labels used elsewhere in the project files.


## 3. ANSYS Geometry-Change Method

The geometry variants were produced in ANSYS SpaceClaim by reducing the heat-sink length from the side opposite the TIM/MOSFET contact region.

For the 60 mm geometry, 20 mm of the original 80 mm heat sink was removed, which also removed two fins.

When the modified geometry replaced the original model, ANSYS initially reported a **missing parts** error. The convection selections and heat-flow boundary conditions were checked and reassigned where required before solving the updated model.

The same overall modelling approach was retained so that temperature differences could be attributed to the geometry/material change rather than to a change in thermal boundary conditions.


## 4. Controlled Simulation Conditions

The following parameters were kept consistent across the geometry comparison:

| Parameter | Value / condition |
|---|---|
| Materials | Aluminium and Copper |
| TIM | TGP5000, fixed configuration |
| Ambient temperature | 25°C |
| Convection | Natural convection, same project boundary condition |
| Baseline heat input | 1.505 W |
| Thermal-stress heat input | 15 W |
| MOSFET junction-to-case resistance | 1.5°C/W |

The **1.505 W** case represents the conservative 10 A baseline electrical heat load. The **15 W** case was added as a deliberately high thermal-stress condition because the baseline geometry/material temperature differences were very small.


## 5. Raw ANSYS Maximum Temperatures

The table below records the raw maximum temperatures reported by ANSYS for the geometry variants.


In [1]:
import pandas as pd

raw_results = pd.DataFrame([
    {"Heat input (W)": 1.505, "Geometry": "60 × 50 × 5 mm", "Material": "Aluminium", "ANSYS max temperature (°C)": 31.455},
    {"Heat input (W)": 1.505, "Geometry": "60 × 50 × 5 mm", "Material": "Copper",    "ANSYS max temperature (°C)": 31.297},
    {"Heat input (W)": 1.505, "Geometry": "40 × 50 × 5 mm", "Material": "Aluminium", "ANSYS max temperature (°C)": 33.217},
    {"Heat input (W)": 1.505, "Geometry": "40 × 50 × 5 mm", "Material": "Copper",    "ANSYS max temperature (°C)": 33.090},
    {"Heat input (W)": 15.000, "Geometry": "40 × 50 × 5 mm", "Material": "Aluminium", "ANSYS max temperature (°C)": 106.900},
    {"Heat input (W)": 15.000, "Geometry": "40 × 50 × 5 mm", "Material": "Copper",    "ANSYS max temperature (°C)": 105.640},
    {"Heat input (W)": 15.000, "Geometry": "60 × 50 × 5 mm", "Material": "Aluminium", "ANSYS max temperature (°C)": 89.331},
    {"Heat input (W)": 15.000, "Geometry": "60 × 50 × 5 mm", "Material": "Copper",    "ANSYS max temperature (°C)": 87.759},
])

raw_results

,Heat input (W),Geometry,Material,ANSYS max temperature (°C)
0,1.505,60 × 50 × 5 mm,Aluminium,31.455
1,1.505,60 × 50 × 5 mm,Copper,31.297
2,1.505,40 × 50 × 5 mm,Aluminium,33.217
3,1.505,40 × 50 × 5 mm,Copper,33.090
4,15.000,40 × 50 × 5 mm,Aluminium,106.900
5,15.000,40 × 50 × 5 mm,Copper,105.640
6,15.000,60 × 50 × 5 mm,Aluminium,89.331
7,15.000,60 × 50 × 5 mm,Copper,87.759


## 6. Junction-Temperature Conversion

The ANSYS maximum temperature represents the simulated package/case-side maximum used in this simplified thermal model. To remain consistent with the rest of the project, the junction temperature is estimated using:

**Estimated Tj = ANSYS maximum temperature + (Heat input × RθJC)**

with **RθJC = 1.5°C/W**.


In [2]:
RTH_JC = 1.5  # °C/W

geometry_results = raw_results.copy()
geometry_results["Junction rise (°C)"] = geometry_results["Heat input (W)"] * RTH_JC
geometry_results["Estimated Tj (°C)"] = (
    geometry_results["ANSYS max temperature (°C)"]
    + geometry_results["Junction rise (°C)"]
)

geometry_results.round(4)

,Heat input (W),Geometry,Material,ANSYS max temperature (°C),Junction rise (°C),Estimated Tj (°C)
0,1.505,60 × 50 × 5 mm,Aluminium,31.455,2.2575,33.7125
1,1.505,60 × 50 × 5 mm,Copper,31.297,2.2575,33.5545
2,1.505,40 × 50 × 5 mm,Aluminium,33.217,2.2575,35.4745
3,1.505,40 × 50 × 5 mm,Copper,33.090,2.2575,35.3475
4,15.000,40 × 50 × 5 mm,Aluminium,106.900,22.5000,129.4000
5,15.000,40 × 50 × 5 mm,Copper,105.640,22.5000,128.1400
6,15.000,60 × 50 × 5 mm,Aluminium,89.331,22.5000,111.8310
7,15.000,60 × 50 × 5 mm,Copper,87.759,22.5000,110.2590


## 7. Geometry Benefit Calculation

The temperature benefit of the larger 60 mm variant relative to the 40 mm variant is calculated separately for aluminium and copper at both heat loads.


In [3]:
comparison_rows = []

for heat_input in [1.505, 15.0]:
    for material in ["Aluminium", "Copper"]:
        subset = geometry_results[
            (geometry_results["Heat input (W)"] == heat_input)
            & (geometry_results["Material"] == material)
        ]

        tj_60 = subset.loc[subset["Geometry"] == "60 × 50 × 5 mm", "Estimated Tj (°C)"].iloc[0]
        tj_40 = subset.loc[subset["Geometry"] == "40 × 50 × 5 mm", "Estimated Tj (°C)"].iloc[0]

        comparison_rows.append({
            "Heat input (W)": heat_input,
            "Material": material,
            "60 mm Tj (°C)": tj_60,
            "40 mm Tj (°C)": tj_40,
            "60 mm geometry benefit (°C)": tj_40 - tj_60,
        })

geometry_comparison = pd.DataFrame(comparison_rows)
geometry_comparison.round(4)

,Heat input (W),Material,60 mm Tj (°C),40 mm Tj (°C),60 mm geometry benefit (°C)
0,1.505,Aluminium,33.7125,35.4745,1.762
1,1.505,Copper,33.5545,35.3475,1.793
2,15.000,Aluminium,111.8310,129.4000,17.569
3,15.000,Copper,110.2590,128.1400,17.881


## 8. Results Interpretation

At the controlled **1.505 W** heat input, the original 80 mm aluminium reference is **32.885°C**, the 60 mm geometry is **33.713°C**, and the 40 mm geometry is **35.475°C**. Reducing from 80 mm to 60 mm therefore removes 25% of the heat-sink mass for only about **0.83°C** additional junction temperature at this fixed heat load.

Comparing the two reduced geometries, the 60 mm heat sink is approximately **1.76°C cooler for aluminium** and **1.79°C cooler for copper** at 1.505 W.

At **15 W**, the same thermal-resistance differences produce a much larger absolute temperature separation: the 60 mm geometry is approximately **17.57°C cooler for aluminium** and **17.88°C cooler for copper** than the 40 mm geometry.

This should not be interpreted as a new nonlinear geometry mechanism appearing at 15 W. Under the approximately linear steady-state assumptions used here, a difference in effective thermal resistance produces a larger absolute temperature penalty as heat load increases. Geometry selection therefore becomes more consequential as the design approaches its thermal constraint.

The material difference remains smaller than the geometry difference, supporting the project sequence of **material screening first, followed by geometry optimisation within aluminium**.


## 9. Modelling Notes and Limitations

- The geometry study compares reduced heat-sink lengths rather than performing a full continuous optimisation of fin dimensions.
- The convection and heat-flow conditions were kept consistent between variants.
- The 15 W case is an imposed thermal-stress case and is **not** an electrically derived MOSFET loss.
- TIM configuration was fixed during the geometry study.
- The 40 mm surface area was not explicitly recorded in the source notes and is therefore not entered here.
- Physical dimensions are preferred over `Geometry_1` / `Geometry_2` labels in this notebook to avoid ambiguity when transferring results into the master case table.
- Because the 80/60/40 mm dimension is across the fin array and the fin count changes, the candidates have different extrusion cross-sections rather than being one common profile cut to length.


## 10. Data Handoff

The processed geometry results should be transferred to `../results/master_results.csv` and used by `../results/engineering_assessment.ipynb` for the final material/geometry trade-off analysis.

This notebook should remain the detailed record of **how the geometry results were generated**, while the engineering-assessment notebook should focus on **what the results mean for the final design decision**.
